# Human Action Classification — Quick Start

This notebook runs the released models end to end:

1. **Image** action recognition (Stanford40, ResNet50) with MediaPipe pose
2. **Pose keypoints** with `PoseExtractor`
3. **Video** action recognition (UCF-101, MC3-18)
4. Pointers for **training** on your own data

Requirements: `pip install -e .` from the repo root (weights are downloaded from the Hugging Face Hub on first use). Runs on CPU or GPU.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch

from hac import ImagePredictor
from hac.image.inference.pose_extractor import PoseExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ASSETS = Path("../assets") if Path("../assets").exists() else Path("assets")
print(f"device={DEVICE}, assets={ASSETS.resolve()}")

## 1. Image action recognition

`model_path` accepts a local `.pth`/`.safetensors` file or a Hub path. Hub folders ship a `config.json`, so the architecture and class names are picked up automatically.

In [ ]:
predictor = ImagePredictor(
    model_path="hf://dronefreak/human-action-classification-stanford40/resnet50",
    device=DEVICE,
    use_pose_estimation=True,
)

image_path = str(ASSETS / "gardening.jpg")
result = predictor.predict_image(image_path, top_k=5)

if "pose" in result:
    print(f"Pose: {result['pose']['class']}")
for i, pred in enumerate(result["action"]["predictions"], 1):
    print(f"{i}. {pred['class']:<30} {pred['confidence']:.3f}")

In [ ]:
# pose_image is BGR (OpenCV); fall back to the input image if no person was found
shown = result.get("pose_image", cv2.imread(image_path))
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
plt.title(f"{result['action']['top_class']} ({result['action']['top_confidence']:.1%})")
plt.axis("off")
plt.show()

## 2. Pose keypoints

MediaPipe returns 33 landmarks as `[x, y, visibility]` in normalized image coordinates. `extract_normalized_keypoints` centres them on the hips and scales by torso length (translation/scale invariant).

In [ ]:
image = cv2.imread(str(ASSETS / "gardening.jpg"))

with PoseExtractor(static_image_mode=True) as extractor:
    keypoints = extractor.extract_keypoints(image)
    normalized = extractor.extract_normalized_keypoints(image)

if keypoints is None:
    print("No person detected")
else:
    print("keypoints:", keypoints.shape, "| normalized:", normalized.shape)
    print("nose [x, y, visibility]:", keypoints[0].round(3))

## 3. Video action recognition

The video pipeline uses a separate 3D-CNN predictor. Point `VIDEO_PATH` at any clip (e.g. a UCF-101 `.avi`); use `num_frames=8` for the Kinetics-initialized HMDB51 model and 16 otherwise.

In [ ]:
from huggingface_hub import hf_hub_download

from hac.video.inference.predictor import VideoPredictor

video_model = hf_hub_download("dronefreak/mc3-18-ucf101", "mc318-ufc101-split-1.pth")
video_predictor = VideoPredictor(model_path=video_model, num_frames=16, device=DEVICE)
print(f"{len(video_predictor.class_names)} classes, e.g. {video_predictor.class_names[:3]}")

VIDEO_PATH = "path/to/clip.avi"  # <- change me
if Path(VIDEO_PATH).exists():
    out = video_predictor.predict_video(VIDEO_PATH, top_k=3)
    for p in out["predictions"]:
        print(f"{p['class']:<25} {p['confidence']:.3f}")
else:
    print(f"Set VIDEO_PATH to a video file to run a prediction ({VIDEO_PATH!r} not found)")

## 4. Training on your own data

Image data is expected as `root/{train,test}/<class>/*.jpg`; videos as `root/{train,test}/<class>/*.avi` (or extracted frame folders). Class indices follow the sorted class-directory names and are stored in every checkpoint.

```bash
# Image (e.g. Stanford40)
python -m hac.image.training.train --data_dir data/stanford40 --model_name resnet50 --num_classes 40

# Video (UCF-101 with the official split files)
python -m hac.video.data.split_ucf101 --source UCF-101/ --output UCF-101-split1/ --splits ucfTrainTestlist/ --split_num 1
python -m hac.video.training.train --data_dir UCF-101-split1/ --model mc3_18 --pretrained
```

In [ ]:
from hac.common.transforms import get_training_transforms
from hac.image.data.dataset import ActionDataset

DATA_ROOT = Path("path/to/your/data")  # <- change me
if (DATA_ROOT / "train").exists():
    train_set = ActionDataset(DATA_ROOT, split="train", transform=get_training_transforms())
    print(f"{len(train_set)} images, classes: {train_set.class_names}")
else:
    print(f"Set DATA_ROOT to a folder containing train/<class>/ images ({DATA_ROOT} not found)")

## Next steps

- `hac-infer --help`, `hac-demo` (Gradio web UI), `python scripts/simple_demo.py --image ...`
- `python -m hac.image.inference.evaluate --help` for per-class metrics and confusion matrices
- Model cards and weights: <https://huggingface.co/dronefreak>